In [ ]:
# BAMS 506 2025W1 BA2 - Optimal Decision Making I
# Final Project
# Group Members:
# Hans
# Model Formulations
# Baseline version (Part A)
# The general idea is to minimize the cost of providing food to children while still meeting their baseline nutritional requirements.

# Let:
# Xi = # of servings of a particular food {i = [1, 2, .., 49]}
# pi = price per serving of a particular food
# mj = variable unit of a particular nutrient {j = [1, 2, … 10]}

# Decision Variables:
# Per the food_data.xlsx table, let the following decision variables equal
# x1 = # of servings of Almonds
# x2 = # of servings of Apple Raw with skin
# …
# x49 = # of servings of White Tuna in Water

# Objective Function:
# Min  Pi*Xi = Sigma[i= 1 to 49](pi*xi) = p1*x1 +p2*x2 + … + p49*x49 = 0.5*x1 + 0.24*x2 + … + 0.69*x49

# Where:
# p1 = price per serving of Almonds
# p2 = price per serving of Apple Raw with skin
# …
# p49 = price per serving of White Tuna in Water

# Constraints:
# Idea behind constraint formulas is to set the upper and lower bound for each nutrient, from a per food serving perspective
# Nutrient Constraints
# Sigma[j = 1 to 10]Sigma[i = 1 to 49]{mj*xi} >= mjMin
# Sigma[j = 1 to 10]Sigma[i = 1 to 49]{mj*xi} <= mjMax

# Written more explicitly:
# m1Min   164*x1 + 81.4*x2 + … + 115.6*x3 >= 1800
# m1Max   164*x1 + 81.4*x2 + … + 115.6*x3 <= 2400
# m2Min   14.2*x1 + 0.5*x2 + … + 2.1*x3 >= 60
# m2Max   14.2*x1 + 0.5*x2 + … + 2.1*x3 <= 95
# …
# m10Min  1.1*x1 + 0.2*x2 + … + 0.5*x3 >= 8
# m10Max  1.1*x1 + 0.2*x2 + … + 0.5*x3 <= 40

# Where:
# m1 = calories measured in kcal
# m2 = fat measured in grams
# …
# m10 = iron measured in milligrams
# Balanced Diet Constraints:
# No single food item should make up more than 30% of total calories OR protein intake

# xi * m1 <=  (0.30) * Sigma[i = 1 to 49](xi * m1) + M * (yi),
# xi * m6 <=  (0.30) * Sigma[i = 1 to 49](xi * m6) + M * (1 - yi)

# Where:
# yi = [0, 1], binary activator
# M = “large constant” to make the value purposely high, in excess of any particular possible max nutrient for a given food, for example M = 10,000

# Non-Negativity
# xi >= 0

In [ ]:
#GenAI Disclaimer: Code was co-created with GenAI (GitHub CoPilot)
# Version 1 - Interpretation of "OR" constraint in balanced diet constraints

import gurobipy as gp
from gurobipy import GRB
import pandas as pd
import numpy as np

# Food data - 49 foods with nutritional information and costs
foods = [
    "Almonds", "Apple Raw wSkin", "Avocado", "Bagels", "Baked Salmon", "Banana", "Beef",
    "Black Beans Canned Drained", "Blueberries", "Broccoli", "Brown Rice Cooked", "Carrots Raw",
    "Cheddar Cheese", "Chickpeas Canned Drained", "Corn", "Couscous", "Grapes", "Greek Yogurt Nonfat",
    "Ham Deli", "Kale Raw", "Kiwifruit Raw Fresh", "Lettuce Iceberg Raw", "Lowfat Milk 2 Percent",
    "Macaroni Cooked", "Mixed Nuts", "Oatmeal", "Oranges", "Peanut Butter", "Peppers Sweet Raw",
    "Poached Eggs", "Pork", "Potatoes Baked", "Quinoa Cooked", "Roasted Chicken", "Scrambled Eggs",
    "Shrimp Cooked", "Skim Milk", "Spaghetti With Sauce", "Spinach Raw", "Strawberries Sliced",
    "Sweet Potato Baked", "Tofu", "Tomato Red Ripe Raw", "Turkey", "Turkey Breast Deli",
    "Wheat Bread", "White Bread", "White Rice", "White Tuna in Water"
]

# Nutritional data matrix [49 foods x 10 nutrients]
# Nutrients: Calories, Fat, Sodium, Carbs, Fiber, Protein, Vitamin A, Vitamin C, Calcium, Iron
nutrition_data = [
    [164, 14.2, 0, 6.1, 3.5, 6, 1, 0, 76, 1.1],
    [81.4, 0.5, 0, 21, 3.7, 0.3, 73.1, 7.9, 9.7, 0.2],
    [120, 11, 5, 6, 5, 1.5, 146, 6, 10, 0.3],
    [78, 0.5, 151.4, 15.1, 0.6, 3, 0, 0, 21, 1],
    [177, 11, 50, 0, 0, 17, 200, 0, 10, 0.7],
    [104.9, 0.5, 1.1, 26.7, 2.7, 1.2, 92.3, 10.4, 6.8, 0.4],
    [141.8, 12.8, 461.7, 0.8, 0, 5.4, 0, 10.8, 9, 0.6],
    [114, 0.5, 230, 20.4, 7.5, 7.6, 0, 0.7, 23, 1.8],
    [84, 0.5, 1, 21, 3.6, 1.1, 80, 14.4, 9, 0.4],
    [73.8, 0.8, 68.2, 13.6, 8.5, 8, 5867.4, 160.2, 159, 2.3],
    [108, 0.9, 5, 22.8, 1.6, 2.3, 0, 0, 10, 0.4],
    [23.7, 0.1, 19.2, 5.6, 1.6, 0.6, 15471, 5.1, 14.9, 0.3],
    [114, 9.4, 174, 0.9, 0, 7, 284, 0, 202, 0.2],
    [135, 2.1, 240, 22.5, 6.2, 7.3, 0, 0, 40, 2.4],
    [72.2, 0.6, 2.5, 17.1, 2, 2.5, 106.6, 5.2, 3.3, 0.3],
    [100.8, 0.1, 4.5, 20.9, 1.3, 3.4, 0, 0, 7.2, 0.3],
    [15.1, 0.1, 0.5, 4.1, 0.2, 0.2, 24, 1, 3.4, 0.1],
    [130, 0.7, 85, 9, 0, 23, 100, 0, 230, 0.1],
    [91, 5, 730, 1.2, 0, 9.5, 0, 0, 9, 0.5],
    [33, 0.6, 25, 6.7, 1.3, 2.2, 10302, 80.4, 90, 1.1],
    [46.4, 0.3, 3.8, 11.3, 2.6, 0.8, 133, 74.5, 19.8, 0.3],
    [2.6, 0, 1.8, 0.4, 0.3, 0.2, 66, 0.8, 3.8, 0.1],
    [121.2, 4.7, 121.8, 11.7, 0, 8.1, 500.2, 2.3, 296.7, 0.1],
    [98.7, 0.5, 0.7, 19.8, 0.9, 3.3, 0, 0, 4.9, 1],
    [170, 15, 0, 6, 2.5, 5, 0, 0, 40, 1],
    [145.1, 2.3, 2.3, 25.3, 4, 6.1, 37.4, 0, 18.7, 1.6],
    [61.6, 0.2, 0, 15.4, 3.1, 1.2, 268.6, 69.7, 52.4, 0.1],
    [188, 16, 152, 6.9, 2.1, 7.7, 0, 0, 17, 0.6],
    [20, 0.1, 1.5, 4.8, 1.3, 0.7, 467.7, 66.1, 6.7, 0.3],
    [74.5, 5, 140, 0.6, 0, 6.2, 316, 0, 24.5, 0.7],
    [710.8, 72.2, 38.4, 0, 0, 13.8, 14.7, 0, 59.9, 0.4],
    [171.5, 0.2, 15.2, 39.9, 3.2, 3.7, 0, 15.6, 22.7, 4.3],
    [111, 1.8, 7, 19.7, 2.6, 4.1, 0, 0, 15, 1.5],
    [277.4, 10.8, 125.6, 0, 0, 42.2, 77.4, 0, 21.9, 1.8],
    [99.6, 7.3, 168, 1.3, 0, 6.7, 409.2, 0.1, 42.6, 0.7],
    [84, 0.9, 190, 0.2, 0, 18, 50, 0, 33, 0.3],
    [85.5, 0.4, 126.2, 11.9, 0, 8.4, 499.8, 2.4, 302.3, 0.1],
    [358.2, 12.3, 1237.1, 58.3, 11.6, 8.2, 3055.2, 27.9, 80.2, 2.3],
    [7, 0.1, 24, 1.1, 0.7, 0.9, 2813, 8.4, 30, 0.8],
    [53, 0.5, 2, 12.7, 3.3, 1.1, 17, 97.6, 27, 0.7],
    [103, 0.2, 41, 23.6, 3.8, 2.3, 21909, 22.3, 43, 0.8],
    [88.2, 5.5, 8.1, 2.2, 1.4, 9.4, 98.6, 0.1, 121.8, 6.2],
    [25.8, 0.4, 11.1, 5.7, 1.4, 1, 766.3, 23.5, 6.2, 0.6],
    [56.4, 4.3, 248.9, 0.3, 0, 3.9, 0, 0, 23.8, 0.4],
    [60, 1, 500, 2, 0, 10, 0, 0, 8, 0.4],
    [65, 1, 134.5, 12.4, 1.3, 2.2, 0, 0, 10.8, 0.7],
    [65, 1, 132.5, 11.8, 1.1, 2.3, 0, 0, 26.2, 0.8],
    [103, 0, 0.2, 22.3, 0.8, 2.1, 0, 0, 7.9, 0.3],
    [115.6, 2.1, 333.2, 0, 0, 22.7, 68, 0, 3.4, 0.5]
]

# Cost per serving for each food
costs = [0.5, 0.24, 0.75, 0.16, 2.5, 0.15, 0.27, 0.3, 1.5, 0.16, 0.1, 0.07, 0.35, 0.35, 0.18,
         0.39, 0.32, 1, 0.7, 0.4, 0.49, 0.02, 0.23, 0.17, 0.6, 0.82, 0.15, 0.25, 0.53, 0.08,
         0.81, 0.06, 0.35, 0.84, 0.11, 1.8, 0.13, 0.78, 0.25, 1.2, 0.5, 0.31, 0.27, 0.15,
         0.8, 0.05, 0.06, 0.08, 0.69]

# Nutrient requirements [min, max] for each nutrient
nutrient_bounds = [
    [1800, 2400],  # Calories (kcal)
    [60, 95],      # Fat (g)
    [1200, 2200],  # Sodium (mg)
    [240, 400],    # Carbs (g)
    [30, 35],      # Fiber (g)
    [40, 55],      # Protein (g)
    [2000, 6000],  # Vitamin A (IU)
    [45, 1200],    # Vitamin C (mg)
    [1300, 3000],  # Calcium (mg)
    [8, 40]        # Iron (mg)
]

# Create optimization model
model = gp.Model("NutritionOptimization")

# Decision variables: number of servings of each food
x = model.addVars(49, name="x", lb=0)

# Binary variables for balanced diet constraints (OR condition)
y = model.addVars(49, vtype=GRB.BINARY, name="y")

# Objective function: minimize total cost
model.setObjective(gp.quicksum(costs[i] * x[i] for i in range(49)), GRB.MINIMIZE)

# Nutrient constraints (min and max for each nutrient)
for j in range(10):  # For each nutrient
    # Minimum requirement
    model.addConstr(gp.quicksum(nutrition_data[i][j] * x[i] for i in range(49)) >= nutrient_bounds[j][0],
                   f"nutrient_{j}_min")
    # Maximum requirement
    model.addConstr(gp.quicksum(nutrition_data[i][j] * x[i] for i in range(49)) <= nutrient_bounds[j][1],
                   f"nutrient_{j}_max")

# Balanced diet constraints: No single food > 30% of total calories OR protein
# Large constant M
M = 10000

for i in range(49):
    # Either calories constraint is active (y[i] = 0) OR protein constraint is active (y[i] = 1)
    # Calories constraint: x[i] * calories[i] <= 0.3 * total_calories + M * y[i]
    model.addConstr(nutrition_data[i][0] * x[i] <=
                   0.3 * gp.quicksum(nutrition_data[k][0] * x[k] for k in range(49)) + M * y[i],
                   f"balanced_calories_{i}")

    # Protein constraint: x[i] * protein[i] <= 0.3 * total_protein + M * (1 - y[i])
    model.addConstr(nutrition_data[i][5] * x[i] <=
                   0.3 * gp.quicksum(nutrition_data[k][5] * x[k] for k in range(49)) + M * (1 - y[i]),
                   f"balanced_protein_{i}")

# Solve the model
model.optimize()

# Display results
if model.status == GRB.OPTIMAL:
    print("OPTIMAL SOLUTION FOUND")
    print("=" * 50)
    print(f"Minimum Cost: ${model.objVal:.2f}")
    print("\nOptimal Food Quantities:")
    print("-" * 50)

    # Show non-zero food quantities
    solution_foods = []
    for i in range(49):
        if x[i].x > 0.001:  # Only show foods with meaningful quantities
            solution_foods.append((foods[i], x[i].x, costs[i] * x[i].x))
            print(f"{foods[i]}: {x[i].x:.3f} servings (${costs[i] * x[i].x:.2f})")

    # Calculate nutritional breakdown
    print("\nNutritional Breakdown:")
    print("-" * 50)
    nutrient_names = ["Calories", "Fat", "Sodium", "Carbs", "Fiber", "Protein",
                     "Vitamin A", "Vitamin C", "Calcium", "Iron"]
    units = ["kcal", "g", "mg", "g", "g", "g", "IU", "mg", "mg", "mg"]

    total_nutrients = [0] * 10
    for j in range(10):
        total_nutrients[j] = sum(nutrition_data[i][j] * x[i].x for i in range(49))
        print(f"{nutrient_names[j]}: {total_nutrients[j]:.1f} {units[j]} " +
              f"(Range: {nutrient_bounds[j][0]}-{nutrient_bounds[j][1]} {units[j]})")

    # Check constraint satisfaction (with small tolerance for floating-point precision)
    print("\nConstraint Satisfaction:")
    print("-" * 50)
    all_satisfied = True
    tolerance = 1e-6  # Small tolerance for floating-point precision issues

    for j in range(10):
        min_req, max_req = nutrient_bounds[j][0], nutrient_bounds[j][1]
        actual = total_nutrients[j]

        # Check with tolerance
        min_satisfied = actual >= (min_req - tolerance)
        max_satisfied = actual <= (max_req + tolerance)
        constraint_satisfied = min_satisfied and max_satisfied

        status = "✓" if constraint_satisfied else "✗"
        if not constraint_satisfied:
            all_satisfied = False

        # Show more detail if close to boundary
        if abs(actual - min_req) < 0.1 or abs(actual - max_req) < 0.1:
            print(f"{nutrient_names[j]}: {status} ({actual:.6f} vs {min_req}-{max_req}) [precise]")
        else:
            print(f"{nutrient_names[j]}: {status} ({actual:.1f} vs {min_req}-{max_req})")

    print(f"\nAll constraints satisfied: {'Yes' if all_satisfied else 'No'}")

    # Check balanced diet constraints (OR logic)
    print("\nBalanced Diet Analysis (OR Constraint Logic):")
    print("-" * 50)
    total_calories = total_nutrients[0]
    total_protein = total_nutrients[5]

    balanced_violations = 0
    constraint_details = []

    for i in range(49):
        if x[i].x > 0.001:  # Only check foods that are actually selected
            cal_pct = (nutrition_data[i][0] * x[i].x) / total_calories * 100
            prot_pct = (nutrition_data[i][5] * x[i].x) / total_protein * 100

            # Get the binary variable value (which constraint is active)
            y_val = y[i].x

            # Check which constraint is active based on binary variable
            if y_val < 0.5:  # y[i] = 0, so calorie constraint is active
                active_constraint = "calories"
                constraint_satisfied = cal_pct <= 30.0
                violated_pct = cal_pct if not constraint_satisfied else None
            else:  # y[i] = 1, so protein constraint is active
                active_constraint = "protein"
                constraint_satisfied = prot_pct <= 30.0
                violated_pct = prot_pct if not constraint_satisfied else None

            if not constraint_satisfied:
                balanced_violations += 1

            constraint_details.append({
                'food': foods[i],
                'servings': x[i].x,
                'cal_pct': cal_pct,
                'prot_pct': prot_pct,
                'active_constraint': active_constraint,
                'satisfied': constraint_satisfied,
                'y_value': y_val
            })

    # Display constraint details for selected foods
    print("Food Item Analysis (Active Constraint | Status):")
    print("-" * 80)
    for detail in constraint_details:
        status_symbol = "✓" if detail['satisfied'] else "✗"
        active_pct = detail['cal_pct'] if detail['active_constraint'] == 'calories' else detail['prot_pct']
        print(f"{detail['food']:<25} | {detail['active_constraint']:<8} | {status_symbol} {active_pct:.1f}% " +
              f"(Cal: {detail['cal_pct']:.1f}%, Prot: {detail['prot_pct']:.1f}%)")

    # Overall balanced diet assessment
    print(f"\nBalanced Diet Constraint Summary:")
    print(f"Total foods violating their active constraint: {balanced_violations}")
    print(f"Balanced diet constraints satisfied: {'Yes' if balanced_violations == 0 else 'No'}")

    # Additional analysis: show max contributors
    max_cal_detail = max(constraint_details, key=lambda x: x['cal_pct']) if constraint_details else None
    max_prot_detail = max(constraint_details, key=lambda x: x['prot_pct']) if constraint_details else None

    if max_cal_detail:
        print(f"\nHighest calorie contributor: {max_cal_detail['food']} ({max_cal_detail['cal_pct']:.1f}%)")
    if max_prot_detail:
        print(f"Highest protein contributor: {max_prot_detail['food']} ({max_prot_detail['prot_pct']:.1f}%)")

else:
    print(f"Optimization failed. Status: {model.status}")
    if model.status == GRB.INFEASIBLE:
        print("The problem is infeasible - no solution exists that satisfies all constraints.")
    elif model.status == GRB.UNBOUNDED:
        print("The problem is unbounded - the objective can be improved indefinitely.")

Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (win64 - Windows 10.0 (19045.2))

CPU model: Intel(R) Core(TM) i7-7700HQ CPU @ 2.80GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 118 rows, 98 columns and 5748 nonzeros
Model fingerprint: 0xd2d4470d
Variable types: 49 continuous, 49 integer (49 binary)
Coefficient statistics:
  Matrix range     [6e-02, 2e+04]
  Objective range  [2e-02, 3e+00]
  Bounds range     [1e+00, 1e+00]

CPU model: Intel(R) Core(TM) i7-7700HQ CPU @ 2.80GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 118 rows, 98 columns and 5748 nonzeros
Model fingerprint: 0xd2d4470d
Variable types: 49 continuous, 49 integer (49 binary)
Coefficient statistics:
  Matrix range     [6e-02, 2e+04]
  Objective range  [2e-02, 3e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [8e+00, 1e+04]
Presolve added

In [ ]:
#GenAI Disclaimer: Code was co-created with GenAI (GitHub CoPilot)
# Version 2: AND Logic for Balanced Diet Constraints, likely more realistic
# This version requires each food to satisfy BOTH calorie AND protein limits (≤30% each)

import gurobipy as gp
from gurobipy import GRB
import pandas as pd
import numpy as np

# Same food data and costs as before
foods = [
    "Almonds", "Apple Raw wSkin", "Avocado", "Bagels", "Baked Salmon", "Banana", "Beef",
    "Black Beans Canned Drained", "Blueberries", "Broccoli", "Brown Rice Cooked", "Carrots Raw",
    "Cheddar Cheese", "Chickpeas Canned Drained", "Corn", "Couscous", "Grapes", "Greek Yogurt Nonfat",
    "Ham Deli", "Kale Raw", "Kiwifruit Raw Fresh", "Lettuce Iceberg Raw", "Lowfat Milk 2 Percent",
    "Macaroni Cooked", "Mixed Nuts", "Oatmeal", "Oranges", "Peanut Butter", "Peppers Sweet Raw",
    "Poached Eggs", "Pork", "Potatoes Baked", "Quinoa Cooked", "Roasted Chicken", "Scrambled Eggs",
    "Shrimp Cooked", "Skim Milk", "Spaghetti With Sauce", "Spinach Raw", "Strawberries Sliced",
    "Sweet Potato Baked", "Tofu", "Tomato Red Ripe Raw", "Turkey", "Turkey Breast Deli",
    "Wheat Bread", "White Bread", "White Rice", "White Tuna in Water"
]

nutrition_data = [
    [164, 14.2, 0, 6.1, 3.5, 6, 1, 0, 76, 1.1],
    [81.4, 0.5, 0, 21, 3.7, 0.3, 73.1, 7.9, 9.7, 0.2],
    [120, 11, 5, 6, 5, 1.5, 146, 6, 10, 0.3],
    [78, 0.5, 151.4, 15.1, 0.6, 3, 0, 0, 21, 1],
    [177, 11, 50, 0, 0, 17, 200, 0, 10, 0.7],
    [104.9, 0.5, 1.1, 26.7, 2.7, 1.2, 92.3, 10.4, 6.8, 0.4],
    [141.8, 12.8, 461.7, 0.8, 0, 5.4, 0, 10.8, 9, 0.6],
    [114, 0.5, 230, 20.4, 7.5, 7.6, 0, 0.7, 23, 1.8],
    [84, 0.5, 1, 21, 3.6, 1.1, 80, 14.4, 9, 0.4],
    [73.8, 0.8, 68.2, 13.6, 8.5, 8, 5867.4, 160.2, 159, 2.3],
    [108, 0.9, 5, 22.8, 1.6, 2.3, 0, 0, 10, 0.4],
    [23.7, 0.1, 19.2, 5.6, 1.6, 0.6, 15471, 5.1, 14.9, 0.3],
    [114, 9.4, 174, 0.9, 0, 7, 284, 0, 202, 0.2],
    [135, 2.1, 240, 22.5, 6.2, 7.3, 0, 0, 40, 2.4],
    [72.2, 0.6, 2.5, 17.1, 2, 2.5, 106.6, 5.2, 3.3, 0.3],
    [100.8, 0.1, 4.5, 20.9, 1.3, 3.4, 0, 0, 7.2, 0.3],
    [15.1, 0.1, 0.5, 4.1, 0.2, 0.2, 24, 1, 3.4, 0.1],
    [130, 0.7, 85, 9, 0, 23, 100, 0, 230, 0.1],
    [91, 5, 730, 1.2, 0, 9.5, 0, 0, 9, 0.5],
    [33, 0.6, 25, 6.7, 1.3, 2.2, 10302, 80.4, 90, 1.1],
    [46.4, 0.3, 3.8, 11.3, 2.6, 0.8, 133, 74.5, 19.8, 0.3],
    [2.6, 0, 1.8, 0.4, 0.3, 0.2, 66, 0.8, 3.8, 0.1],
    [121.2, 4.7, 121.8, 11.7, 0, 8.1, 500.2, 2.3, 296.7, 0.1],
    [98.7, 0.5, 0.7, 19.8, 0.9, 3.3, 0, 0, 4.9, 1],
    [170, 15, 0, 6, 2.5, 5, 0, 0, 40, 1],
    [145.1, 2.3, 2.3, 25.3, 4, 6.1, 37.4, 0, 18.7, 1.6],
    [61.6, 0.2, 0, 15.4, 3.1, 1.2, 268.6, 69.7, 52.4, 0.1],
    [188, 16, 152, 6.9, 2.1, 7.7, 0, 0, 17, 0.6],
    [20, 0.1, 1.5, 4.8, 1.3, 0.7, 467.7, 66.1, 6.7, 0.3],
    [74.5, 5, 140, 0.6, 0, 6.2, 316, 0, 24.5, 0.7],
    [710.8, 72.2, 38.4, 0, 0, 13.8, 14.7, 0, 59.9, 0.4],
    [171.5, 0.2, 15.2, 39.9, 3.2, 3.7, 0, 15.6, 22.7, 4.3],
    [111, 1.8, 7, 19.7, 2.6, 4.1, 0, 0, 15, 1.5],
    [277.4, 10.8, 125.6, 0, 0, 42.2, 77.4, 0, 21.9, 1.8],
    [99.6, 7.3, 168, 1.3, 0, 6.7, 409.2, 0.1, 42.6, 0.7],
    [84, 0.9, 190, 0.2, 0, 18, 50, 0, 33, 0.3],
    [85.5, 0.4, 126.2, 11.9, 0, 8.4, 499.8, 2.4, 302.3, 0.1],
    [358.2, 12.3, 1237.1, 58.3, 11.6, 8.2, 3055.2, 27.9, 80.2, 2.3],
    [7, 0.1, 24, 1.1, 0.7, 0.9, 2813, 8.4, 30, 0.8],
    [53, 0.5, 2, 12.7, 3.3, 1.1, 17, 97.6, 27, 0.7],
    [103, 0.2, 41, 23.6, 3.8, 2.3, 21909, 22.3, 43, 0.8],
    [88.2, 5.5, 8.1, 2.2, 1.4, 9.4, 98.6, 0.1, 121.8, 6.2],
    [25.8, 0.4, 11.1, 5.7, 1.4, 1, 766.3, 23.5, 6.2, 0.6],
    [56.4, 4.3, 248.9, 0.3, 0, 3.9, 0, 0, 23.8, 0.4],
    [60, 1, 500, 2, 0, 10, 0, 0, 8, 0.4],
    [65, 1, 134.5, 12.4, 1.3, 2.2, 0, 0, 10.8, 0.7],
    [65, 1, 132.5, 11.8, 1.1, 2.3, 0, 0, 26.2, 0.8],
    [103, 0, 0.2, 22.3, 0.8, 2.1, 0, 0, 7.9, 0.3],
    [115.6, 2.1, 333.2, 0, 0, 22.7, 68, 0, 3.4, 0.5]
]

costs = [0.5, 0.24, 0.75, 0.16, 2.5, 0.15, 0.27, 0.3, 1.5, 0.16, 0.1, 0.07, 0.35, 0.35, 0.18,
         0.39, 0.32, 1, 0.7, 0.4, 0.49, 0.02, 0.23, 0.17, 0.6, 0.82, 0.15, 0.25, 0.53, 0.08,
         0.81, 0.06, 0.35, 0.84, 0.11, 1.8, 0.13, 0.78, 0.25, 1.2, 0.5, 0.31, 0.27, 0.15,
         0.8, 0.05, 0.06, 0.08, 0.69]

nutrient_bounds = [
    [1800, 2400],  # Calories (kcal)
    [60, 95],      # Fat (g)
    [1200, 2200],  # Sodium (mg)
    [240, 400],    # Carbs (g)
    [30, 35],      # Fiber (g)
    [40, 55],      # Protein (g)
    [2000, 6000],  # Vitamin A (IU)
    [45, 1200],    # Vitamin C (mg)
    [1300, 3000],  # Calcium (mg)
    [8, 40]        # Iron (mg)
]

# Create optimization model (AND logic version)
model_and = gp.Model("NutritionOptimization_AND")

# Decision variables: number of servings of each food (NO binary variables needed)
x_and = model_and.addVars(49, name="x", lb=0)

# Objective function: minimize total cost
model_and.setObjective(gp.quicksum(costs[i] * x_and[i] for i in range(49)), GRB.MINIMIZE)

# Nutrient constraints (same as before)
for j in range(10):
    model_and.addConstr(gp.quicksum(nutrition_data[i][j] * x_and[i] for i in range(49)) >= nutrient_bounds[j][0],
                       f"nutrient_{j}_min")
    model_and.addConstr(gp.quicksum(nutrition_data[i][j] * x_and[i] for i in range(49)) <= nutrient_bounds[j][1],
                       f"nutrient_{j}_max")

# Balanced diet constraints (AND logic): Each food must satisfy BOTH constraints
for i in range(49):
    # Calorie constraint: x[i] * calories[i] <= 0.3 * total_calories
    model_and.addConstr(nutrition_data[i][0] * x_and[i] <=
                       0.3 * gp.quicksum(nutrition_data[k][0] * x_and[k] for k in range(49)),
                       f"balanced_calories_{i}")

    # Protein constraint: x[i] * protein[i] <= 0.3 * total_protein
    model_and.addConstr(nutrition_data[i][5] * x_and[i] <=
                       0.3 * gp.quicksum(nutrition_data[k][5] * x_and[k] for k in range(49)),
                       f"balanced_protein_{i}")

# Solve the model
print("SOLVING AND LOGIC VERSION...")
print("=" * 60)
model_and.optimize()

# Display results
if model_and.status == GRB.OPTIMAL:
    print("OPTIMAL SOLUTION FOUND (AND Logic)")
    print("=" * 50)
    print(f"Minimum Cost: ${model_and.objVal:.2f}")
    print("\nOptimal Food Quantities:")
    print("-" * 50)

    # Show non-zero food quantities
    solution_foods_and = []
    for i in range(49):
        if x_and[i].x > 0.001:
            solution_foods_and.append((foods[i], x_and[i].x, costs[i] * x_and[i].x))
            print(f"{foods[i]}: {x_and[i].x:.3f} servings (${costs[i] * x_and[i].x:.2f})")

    # Calculate nutritional breakdown
    print("\nNutritional Breakdown:")
    print("-" * 50)
    nutrient_names = ["Calories", "Fat", "Sodium", "Carbs", "Fiber", "Protein",
                     "Vitamin A", "Vitamin C", "Calcium", "Iron"]
    units = ["kcal", "g", "mg", "g", "g", "g", "IU", "mg", "mg", "mg"]

    total_nutrients_and = [0] * 10
    for j in range(10):
        total_nutrients_and[j] = sum(nutrition_data[i][j] * x_and[i].x for i in range(49))
        print(f"{nutrient_names[j]}: {total_nutrients_and[j]:.1f} {units[j]} " +
              f"(Range: {nutrient_bounds[j][0]}-{nutrient_bounds[j][1]} {units[j]})")

    # Check constraint satisfaction (with tolerance for floating-point precision)
    print("\nConstraint Satisfaction:")
    print("-" * 50)
    all_satisfied_and = True
    tolerance = 1e-6  # Small tolerance for floating-point precision issues

    for j in range(10):
        min_req, max_req = nutrient_bounds[j][0], nutrient_bounds[j][1]
        actual = total_nutrients_and[j]

        # Check with tolerance
        min_satisfied = actual >= (min_req - tolerance)
        max_satisfied = actual <= (max_req + tolerance)
        constraint_satisfied = min_satisfied and max_satisfied

        status = "✓" if constraint_satisfied else "✗"
        if not constraint_satisfied:
            all_satisfied_and = False

        # Show more detail if close to boundary
        if abs(actual - min_req) < 0.1 or abs(actual - max_req) < 0.1:
            print(f"{nutrient_names[j]}: {status} ({actual:.6f} vs {min_req}-{max_req}) [precise]")
        else:
            print(f"{nutrient_names[j]}: {status} ({actual:.1f} vs {min_req}-{max_req})")

    print(f"\nAll constraints satisfied: {'Yes' if all_satisfied_and else 'No'}")

    # Check balanced diet constraints (AND logic)
    print("\nBalanced Diet Analysis (AND Constraint Logic):")
    print("-" * 50)
    total_calories_and = total_nutrients_and[0]
    total_protein_and = total_nutrients_and[5]

    calorie_violations = 0
    protein_violations = 0
    both_satisfied = 0

    print("Food Item Analysis (Both Constraints Must Be Satisfied):")
    print("-" * 80)
    print(f"{'Food Name':<25} | {'Cal %':<8} | {'Prot %':<8} | {'Status':<15}")
    print("-" * 80)

    for i in range(49):
        if x_and[i].x > 0.001:
            cal_pct = (nutrition_data[i][0] * x_and[i].x) / total_calories_and * 100
            prot_pct = (nutrition_data[i][5] * x_and[i].x) / total_protein_and * 100

            cal_ok = cal_pct <= 30.0
            prot_ok = prot_pct <= 30.0

            if not cal_ok:
                calorie_violations += 1
            if not prot_ok:
                protein_violations += 1
            if cal_ok and prot_ok:
                both_satisfied += 1

            # Status symbols
            cal_symbol = "✓" if cal_ok else "✗"
            prot_symbol = "✓" if prot_ok else "✗"
            overall_status = "PASS" if (cal_ok and prot_ok) else "FAIL"

            print(f"{foods[i]:<25} | {cal_symbol} {cal_pct:>5.1f}% | {prot_symbol} {prot_pct:>5.1f}% | {overall_status}")

    # Overall assessment
    print(f"\nBalanced Diet Constraint Summary (AND Logic):")
    print(f"Foods violating calorie limit (>30%): {calorie_violations}")
    print(f"Foods violating protein limit (>30%): {protein_violations}")
    print(f"Foods satisfying both constraints: {both_satisfied}")
    print(f"Balanced diet constraints satisfied: {'Yes' if calorie_violations == 0 and protein_violations == 0 else 'No'}")

    # Show max contributors
    if solution_foods_and:
        max_cal_pct = 0
        max_prot_pct = 0
        max_cal_food = ""
        max_prot_food = ""

        for i in range(49):
            if x_and[i].x > 0.001:
                cal_pct = (nutrition_data[i][0] * x_and[i].x) / total_calories_and * 100
                prot_pct = (nutrition_data[i][5] * x_and[i].x) / total_protein_and * 100

                if cal_pct > max_cal_pct:
                    max_cal_pct = cal_pct
                    max_cal_food = foods[i]

                if prot_pct > max_prot_pct:
                    max_prot_pct = prot_pct
                    max_prot_food = foods[i]

        print(f"\nHighest calorie contributor: {max_cal_food} ({max_cal_pct:.1f}%)")
        print(f"Highest protein contributor: {max_prot_food} ({max_prot_pct:.1f}%)")

else:
    print(f"Optimization failed. Status: {model_and.status}")
    if model_and.status == GRB.INFEASIBLE:
        print("The problem is infeasible - no solution exists that satisfies all constraints.")
        print("This suggests the AND logic constraints may be too restrictive.")
    elif model_and.status == GRB.UNBOUNDED:
        print("The problem is unbounded - the objective can be improved indefinitely.")

SOLVING AND LOGIC VERSION...
Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (win64 - Windows 10.0 (19045.2))

CPU model: Intel(R) Core(TM) i7-7700HQ CPU @ 2.80GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 118 rows, 49 columns and 5650 nonzeros
Model fingerprint: 0x659da352
Coefficient statistics:
  Matrix range     [6e-02, 2e+04]
  Objective range  [2e-02, 3e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [8e+00, 6e+03]

CPU model: Intel(R) Core(TM) i7-7700HQ CPU @ 2.80GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 118 rows, 49 columns and 5650 nonzeros
Model fingerprint: 0x659da352
Coefficient statistics:
  Matrix range     [6e-02, 2e+04]
  Objective range  [2e-02, 3e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [8e+00, 6e+03]
Presolve removed 10 rows and 0 columns
Presolve time: 0.01s

In [ ]:
#GenAI Disclaimer: Code was co-created with GenAI (GitHub CoPilot)
# Version 3: AND Logic for Balanced Diet Constraints, with Food Group Constraints
# This version uses food groups as defined in the provided mapping (From updated excel sheet)

import gurobipy as gp
from gurobipy import GRB

foods = [
    "Almonds", "Apple Raw wSkin", "Avocado", "Bagels", "Baked Salmon", "Banana", "Beef",
    "Black Beans Canned Drained", "Blueberries", "Broccoli", "Brown Rice Cooked", "Carrots Raw",
    "Cheddar Cheese", "Chickpeas Canned Drained", "Corn", "Couscous", "Grapes", "Greek Yogurt Nonfat",
    "Ham Deli", "Kale Raw", "Kiwifruit Raw Fresh", "Lettuce Iceberg Raw", "Lowfat Milk 2 Percent",
    "Macaroni Cooked", "Mixed Nuts", "Oatmeal", "Oranges", "Peanut Butter", "Peppers Sweet Raw",
    "Poached Eggs", "Pork", "Potatoes Baked", "Quinoa Cooked", "Roasted Chicken", "Scrambled Eggs",
    "Shrimp Cooked", "Skim Milk", "Spaghetti With Sauce", "Spinach Raw", "Strawberries Sliced",
    "Sweet Potato Baked", "Tofu", "Tomato Red Ripe Raw", "Turkey", "Turkey Breast Deli",
    "Wheat Bread", "White Bread", "White Rice", "White Tuna in Water"
]

nutrition_data = [
    [164, 14.2, 0, 6.1, 3.5, 6, 1, 0, 76, 1.1],
    [81.4, 0.5, 0, 21, 3.7, 0.3, 73.1, 7.9, 9.7, 0.2],
    [120, 11, 5, 6, 5, 1.5, 146, 6, 10, 0.3],
    [78, 0.5, 151.4, 15.1, 0.6, 3, 0, 0, 21, 1],
    [177, 11, 50, 0, 0, 17, 200, 0, 10, 0.7],
    [104.9, 0.5, 1.1, 26.7, 2.7, 1.2, 92.3, 10.4, 6.8, 0.4],
    [141.8, 12.8, 461.7, 0.8, 0, 5.4, 0, 10.8, 9, 0.6],
    [114, 0.5, 230, 20.4, 7.5, 7.6, 0, 0.7, 23, 1.8],
    [84, 0.5, 1, 21, 3.6, 1.1, 80, 14.4, 9, 0.4],
    [73.8, 0.8, 68.2, 13.6, 8.5, 8, 5867.4, 160.2, 159, 2.3],
    [108, 0.9, 5, 22.8, 1.6, 2.3, 0, 0, 10, 0.4],
    [23.7, 0.1, 19.2, 5.6, 1.6, 0.6, 15471, 5.1, 14.9, 0.3],
    [114, 9.4, 174, 0.9, 0, 7, 284, 0, 202, 0.2],
    [135, 2.1, 240, 22.5, 6.2, 7.3, 0, 0, 40, 2.4],
    [72.2, 0.6, 2.5, 17.1, 2, 2.5, 106.6, 5.2, 3.3, 0.3],
    [100.8, 0.1, 4.5, 20.9, 1.3, 3.4, 0, 0, 7.2, 0.3],
    [15.1, 0.1, 0.5, 4.1, 0.2, 0.2, 24, 1, 3.4, 0.1],
    [130, 0.7, 85, 9, 0, 23, 100, 0, 230, 0.1],
    [91, 5, 730, 1.2, 0, 9.5, 0, 0, 9, 0.5],
    [33, 0.6, 25, 6.7, 1.3, 2.2, 10302, 80.4, 90, 1.1],
    [46.4, 0.3, 3.8, 11.3, 2.6, 0.8, 133, 74.5, 19.8, 0.3],
    [2.6, 0, 1.8, 0.4, 0.3, 0.2, 66, 0.8, 3.8, 0.1],
    [121.2, 4.7, 121.8, 11.7, 0, 8.1, 500.2, 2.3, 296.7, 0.1],
    [98.7, 0.5, 0.7, 19.8, 0.9, 3.3, 0, 0, 4.9, 1],
    [170, 15, 0, 6, 2.5, 5, 0, 0, 40, 1],
    [145.1, 2.3, 2.3, 25.3, 4, 6.1, 37.4, 0, 18.7, 1.6],
    [61.6, 0.2, 0, 15.4, 3.1, 1.2, 268.6, 69.7, 52.4, 0.1],
    [188, 16, 152, 6.9, 2.1, 7.7, 0, 0, 17, 0.6],
    [20, 0.1, 1.5, 4.8, 1.3, 0.7, 467.7, 66.1, 6.7, 0.3],
    [74.5, 5, 140, 0.6, 0, 6.2, 316, 0, 24.5, 0.7],
    [710.8, 72.2, 38.4, 0, 0, 13.8, 14.7, 0, 59.9, 0.4],
    [171.5, 0.2, 15.2, 39.9, 3.2, 3.7, 0, 15.6, 22.7, 4.3],
    [111, 1.8, 7, 19.7, 2.6, 4.1, 0, 0, 15, 1.5],
    [277.4, 10.8, 125.6, 0, 0, 42.2, 77.4, 0, 21.9, 1.8],
    [99.6, 7.3, 168, 1.3, 0, 6.7, 409.2, 0.1, 42.6, 0.7],
    [84, 0.9, 190, 0.2, 0, 18, 50, 0, 33, 0.3],
    [85.5, 0.4, 126.2, 11.9, 0, 8.4, 499.8, 2.4, 302.3, 0.1],
    [358.2, 12.3, 1237.1, 58.3, 11.6, 8.2, 3055.2, 27.9, 80.2, 2.3],
    [7, 0.1, 24, 1.1, 0.7, 0.9, 2813, 8.4, 30, 0.8],
    [53, 0.5, 2, 12.7, 3.3, 1.1, 17, 97.6, 27, 0.7],
    [103, 0.2, 41, 23.6, 3.8, 2.3, 21909, 22.3, 43, 0.8],
    [88.2, 5.5, 8.1, 2.2, 1.4, 9.4, 98.6, 0.1, 121.8, 6.2],
    [25.8, 0.4, 11.1, 5.7, 1.4, 1, 766.3, 23.5, 6.2, 0.6],
    [56.4, 4.3, 248.9, 0.3, 0, 3.9, 0, 0, 23.8, 0.4],
    [60, 1, 500, 2, 0, 10, 0, 0, 8, 0.4],
    [65, 1, 134.5, 12.4, 1.3, 2.2, 0, 0, 10.8, 0.7],
    [65, 1, 132.5, 11.8, 1.1, 2.3, 0, 0, 26.2, 0.8],
    [103, 0, 0.2, 22.3, 0.8, 2.1, 0, 0, 7.9, 0.3],
    [115.6, 2.1, 333.2, 0, 0, 22.7, 68, 0, 3.4, 0.5]
]

costs = [0.5, 0.24, 0.75, 0.16, 2.5, 0.15, 0.27, 0.3, 1.5, 0.16, 0.1, 0.07, 0.35, 0.35, 0.18,
         0.39, 0.32, 1, 0.7, 0.4, 0.49, 0.02, 0.23, 0.17, 0.6, 0.82, 0.15, 0.25, 0.53, 0.08,
         0.81, 0.06, 0.35, 0.84, 0.11, 1.8, 0.13, 0.78, 0.25, 1.2, 0.5, 0.31, 0.27, 0.15,
         0.8, 0.05, 0.06, 0.08, 0.69]

nutrient_bounds = [
    [1800, 2400],  # Calories (kcal)
    [60, 95],      # Fat (g)
    [1200, 2200],  # Sodium (mg)
    [240, 400],    # Carbs (g)
    [30, 35],      # Fiber (g)
    [40, 55],      # Protein (g)
    [2000, 6000],  # Vitamin A (IU)
    [45, 1200],    # Vitamin C (mg)
    [1300, 3000],  # Calcium (mg)
    [8, 40]        # Iron (mg)
]

# --- Food Groups from Excel Mapping ---
food_group_labels = [
    "Protein", "Fruit", "Fruit", "Grain/Starch", "Protein", "Fruit", "Protein", "Protein", "Fruit", "Vegetable",
    "Grain/Starch", "Vegetable", "Dairy", "Protein", "Vegetable", "Grain/Starch", "Fruit", "Dairy", "Protein", "Vegetable",
    "Fruit", "Vegetable", "Dairy", "Grain/Starch", "Grain/Starch", "Grain/Starch", "Fruit", "Protein", "Vegetable", "Protein",
    "Protein", "Vegetable", "Grain/Starch", "Protein", "Protein", "Protein", "Protein", "Grain/Starch", "Vegetable", "Fruit",
    "Vegetable", "Protein", "Vegetable", "Protein", "Protein", "Grain/Starch", "Grain/Starch", "Grain/Starch", "Protein"
]

# Build food_groups dictionary from labels
food_groups = {}
for idx, label in enumerate(food_group_labels):
    if label not in food_groups:
        food_groups[label] = []
    food_groups[label].append(idx)

# --- End Food Groups from Excel Mapping ---

# Create optimization model (AND logic version)
model_and = gp.Model("NutritionOptimization_AND")

# Decision variables: number of servings of each food (NO binary variables needed)
x_and = model_and.addVars(49, name="x", lb=0)

# Objective function: minimize total cost
model_and.setObjective(gp.quicksum(costs[i] * x_and[i] for i in range(49)), GRB.MINIMIZE)

# Nutrient constraints (same as before)
for j in range(10):
    model_and.addConstr(gp.quicksum(nutrition_data[i][j] * x_and[i] for i in range(49)) >= nutrient_bounds[j][0],
                       f"nutrient_{j}_min")
    model_and.addConstr(gp.quicksum(nutrition_data[i][j] * x_and[i] for i in range(49)) <= nutrient_bounds[j][1],
                       f"nutrient_{j}_max")

# Balanced diet constraints (AND logic): Each food must satisfy BOTH constraints
for i in range(49):
    # Calorie constraint: x[i] * calories[i] <= 0.3 * total_calories
    model_and.addConstr(nutrition_data[i][0] * x_and[i] <=
                       0.3 * gp.quicksum(nutrition_data[k][0] * x_and[k] for k in range(49)),
                       f"balanced_calories_{i}")
    # Protein constraint: x[i] * protein[i] <= 0.3 * total_protein
    model_and.addConstr(nutrition_data[i][5] * x_and[i] <=
                       0.3 * gp.quicksum(nutrition_data[k][5] * x_and[k] for k in range(49)),
                       f"balanced_protein_{i}")

# --- Food Group Constraints ---
# For each food group, require total servings in specified range
for group_name, indices in food_groups.items():
    if group_name in ["Fruit", "Vegetable"]:
        # Require between 2 and 4 servings (inclusive)
        model_and.addConstr(gp.quicksum(x_and[i] for i in indices) >= 2, f"foodgroup_{group_name}_min2")
        model_and.addConstr(gp.quicksum(x_and[i] for i in indices) <= 4, f"foodgroup_{group_name}_max4")
    else:
        model_and.addConstr(gp.quicksum(x_and[i] for i in indices) >= 1, f"foodgroup_{group_name}_min1")
# --- End Food Group Constraints ---

# Solve the model
print("SOLVING AND LOGIC VERSION WITH FOOD GROUP CONSTRAINTS (Excel Mapping)...")
print("=" * 60)
model_and.optimize()

# Display results
if model_and.status == GRB.OPTIMAL:
    print("OPTIMAL SOLUTION FOUND (AND Logic + Food Groups)")
    print("=" * 50)
    print(f"Minimum Cost: ${model_and.objVal:.2f}")
    print("\nOptimal Food Quantities:")
    print("-" * 50)
    # Show non-zero food quantities
    solution_foods_and = []
    for i in range(49):
        if x_and[i].x > 0.001:
            solution_foods_and.append((foods[i], x_and[i].x, costs[i] * x_and[i].x))
            print(f"{foods[i]}: {x_and[i].x:.3f} servings (${costs[i] * x_and[i].x:.2f})")
    # Calculate nutritional breakdown
    print("\nNutritional Breakdown:")
    print("-" * 50)
    nutrient_names = ["Calories", "Fat", "Sodium", "Carbs", "Fiber", "Protein",
                     "Vitamin A", "Vitamin C", "Calcium", "Iron"]
    units = ["kcal", "g", "mg", "g", "g", "g", "IU", "mg", "mg", "mg"]
    total_nutrients_and = [0] * 10
    for j in range(10):
        total_nutrients_and[j] = sum(nutrition_data[i][j] * x_and[i].x for i in range(49))
        print(f"{nutrient_names[j]}: {total_nutrients_and[j]:.1f} {units[j]} " +
              f"(Range: {nutrient_bounds[j][0]}-{nutrient_bounds[j][1]} {units[j]})")
    # Check constraint satisfaction (with tolerance for floating-point precision)
    print("\nConstraint Satisfaction:")
    print("-" * 50)
    all_satisfied_and = True
    tolerance = 1e-6  # Small tolerance for floating-point precision issues
    for j in range(10):
        min_req, max_req = nutrient_bounds[j][0], nutrient_bounds[j][1]
        actual = total_nutrients_and[j]
        min_satisfied = actual >= (min_req - tolerance)
        max_satisfied = actual <= (max_req + tolerance)
        constraint_satisfied = min_satisfied and max_satisfied
        status = "✓" if constraint_satisfied else "✗"
        if not constraint_satisfied:
            all_satisfied_and = False
        if abs(actual - min_req) < 0.1 or abs(actual - max_req) < 0.1:
            print(f"{nutrient_names[j]}: {status} ({actual:.6f} vs {min_req}-{max_req}) [precise]")
        else:
            print(f"{nutrient_names[j]}: {status} ({actual:.1f} vs {min_req}-{max_req})")
    print(f"\nAll constraints satisfied: {'Yes' if all_satisfied_and else 'No'}")
    # Check balanced diet constraints (AND logic)
    print("\nBalanced Diet Analysis (AND Constraint Logic):")
    print("-" * 50)
    total_calories_and = total_nutrients_and[0]
    total_protein_and = total_nutrients_and[5]
    calorie_violations = 0
    protein_violations = 0
    both_satisfied = 0
    print("Food Item Analysis (Both Constraints Must Be Satisfied):")
    print("-" * 80)
    print(f"{'Food Name':<25} | {'Cal %':<8} | {'Prot %':<8} | {'Status':<15}")
    print("-" * 80)
    for i in range(49):
        if x_and[i].x > 0.001:
            cal_pct = (nutrition_data[i][0] * x_and[i].x) / total_calories_and * 100
            prot_pct = (nutrition_data[i][5] * x_and[i].x) / total_protein_and * 100
            cal_ok = cal_pct <= 30.0
            prot_ok = prot_pct <= 30.0
            if not cal_ok:
                calorie_violations += 1
            if not prot_ok:
                protein_violations += 1
            if cal_ok and prot_ok:
                both_satisfied += 1
            cal_symbol = "✓" if cal_ok else "✗"
            prot_symbol = "✓" if prot_ok else "✗"
            overall_status = "PASS" if (cal_ok and prot_ok) else "FAIL"
            print(f"{foods[i]:<25} | {cal_symbol} {cal_pct:>5.1f}% | {prot_symbol} {prot_pct:>5.1f}% | {overall_status}")
    print(f"\nBalanced Diet Constraint Summary (AND Logic):")
    print(f"Foods violating calorie limit (>30%): {calorie_violations}")
    print(f"Foods violating protein limit (>30%): {protein_violations}")
    print(f"Foods satisfying both constraints: {both_satisfied}")
    print(f"Balanced diet constraints satisfied: {'Yes' if calorie_violations == 0 and protein_violations == 0 else 'No'}")
    # Show max contributors
    if solution_foods_and:
        max_cal_pct = 0
        max_prot_pct = 0
        max_cal_food = ""
        max_prot_food = ""
        for i in range(49):
            if x_and[i].x > 0.001:
                cal_pct = (nutrition_data[i][0] * x_and[i].x) / total_calories_and * 100
                prot_pct = (nutrition_data[i][5] * x_and[i].x) / total_protein_and * 100
                if cal_pct > max_cal_pct:
                    max_cal_pct = cal_pct
                    max_cal_food = foods[i]
                if prot_pct > max_prot_pct:
                    max_prot_pct = prot_pct
                    max_prot_food = foods[i]
        print(f"\nHighest calorie contributor: {max_cal_food} ({max_cal_pct:.1f}%)")
        print(f"Highest protein contributor: {max_prot_food} ({max_prot_pct:.1f}%)")
    # Output servings per food group after optimization
    print("\nServings per Food Group:")
    print("-" * 50)
    for group_name, indices in food_groups.items():
        total_servings = sum(x_and[i].x for i in indices)
        print(f"{group_name}: {total_servings:.3f} servings")
else:
    print(f"Optimization failed. Status: {model_and.status}")
    if model_and.status == GRB.INFEASIBLE:
        print("The problem is infeasible - no solution exists that satisfies all constraints.")
        print("This suggests the AND logic constraints may be too restrictive.")
    elif model_and.status == GRB.UNBOUNDED:
        print("The problem is unbounded - the objective can be improved indefinitely.")


SOLVING AND LOGIC VERSION WITH FOOD GROUP CONSTRAINTS (Excel Mapping)...
Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (win64 - Windows 11.0 (26100.2))

CPU model: 13th Gen Intel(R) Core(TM) i5-1335U, instruction set [SSE2|AVX|AVX2]
Thread count: 10 physical cores, 12 logical processors, using up to 12 threads

Optimize a model with 125 rows, 49 columns and 5717 nonzeros
Model fingerprint: 0x3e41e0f5
Coefficient statistics:
  Matrix range     [6e-02, 2e+04]
  Objective range  [2e-02, 3e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+00, 6e+03]
Presolve removed 12 rows and 0 columns
Presolve time: 0.00s
Presolved: 113 rows, 61 columns, 5287 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    0.0000000e+00   2.614375e+02   0.000000e+00      0s
      22    2.6496835e+00   0.000000e+00   0.000000e+00      0s

Solved in 22 iterations and 0.01 seconds (0.00 work units)
Optimal objective  2.649683451e+00
OPTIMAL SOLUTION FOUND (AND Logic + Fo

In [ ]:
#GenAI Disclaimer: Code was co-created with GenAI (GitHub Copilot)
# Version 4: 5-Day Plan with Limited Repetition (max 2 appearances per food per 5 days)

import gurobipy as gp
from gurobipy import GRB

DAYS = 5
NFOODS = 49

# Decision variables: servings of food i on day d
model_5day = gp.Model("NutritionOptimization_5Day_LimitedRepetition")
x_5day = model_5day.addVars(DAYS, NFOODS, name="x", lb=0)

# Binary variables: 1 if food i is used on day d, 0 otherwise
appear = model_5day.addVars(DAYS, NFOODS, vtype=GRB.BINARY, name="appear")

# Objective: minimize total cost over 5 days
model_5day.setObjective(
    gp.quicksum(costs[i] * x_5day[d, i] for d in range(DAYS) for i in range(NFOODS)),
    GRB.MINIMIZE
)

# --- Per-day constraints ---
for d in range(DAYS):
    # Nutrient constraints
    for j in range(10):
        model_5day.addConstr(
            gp.quicksum(nutrition_data[i][j] * x_5day[d, i] for i in range(NFOODS)) >= nutrient_bounds[j][0],
            f"nutrient_{j}_min_day{d}"
        )
        model_5day.addConstr(
            gp.quicksum(nutrition_data[i][j] * x_5day[d, i] for i in range(NFOODS)) <= nutrient_bounds[j][1],
            f"nutrient_{j}_max_day{d}"
        )
    # Balanced diet constraints (AND logic)
    total_calories = gp.quicksum(nutrition_data[i][0] * x_5day[d, i] for i in range(NFOODS))
    total_protein = gp.quicksum(nutrition_data[i][5] * x_5day[d, i] for i in range(NFOODS))
    for i in range(NFOODS):
        model_5day.addConstr(
            nutrition_data[i][0] * x_5day[d, i] <= 0.3 * total_calories,
            f"balanced_calories_{i}_day{d}"
        )
        model_5day.addConstr(
            nutrition_data[i][5] * x_5day[d, i] <= 0.3 * total_protein,
            f"balanced_protein_{i}_day{d}"
        )
    # Food group constraints
    for group_name, indices in food_groups.items():
        if group_name in ["Fruit", "Vegetable"]:
            model_5day.addConstr(
                gp.quicksum(x_5day[d, i] for i in indices) >= 2,
                f"foodgroup_{group_name}_min2_day{d}"
            )
            model_5day.addConstr(
                gp.quicksum(x_5day[d, i] for i in indices) <= 4,
                f"foodgroup_{group_name}_max4_day{d}"
            )
        else:
            model_5day.addConstr(
                gp.quicksum(x_5day[d, i] for i in indices) >= 1,
                f"foodgroup_{group_name}_min1_day{d}"
            )

# --- Limited repetition constraint: each food appears at most 2 times in 5 days ---
for i in range(NFOODS):
    model_5day.addConstr(
        gp.quicksum(appear[d, i] for d in range(DAYS)) <= 2,
        f"limit_repetition_{i}"
    )

# --- Link x_5day and appear: if x_5day[d, i] > 0 then appear[d, i] = 1 ---
BIG_M = 1000  # Large enough upper bound for servings
for d in range(DAYS):
    for i in range(NFOODS):
        model_5day.addConstr(
            x_5day[d, i] <= BIG_M * appear[d, i],
            f"link_x_appear_{d}_{i}"
        )

print("SOLVING 5-DAY PLAN WITH LIMITED REPETITION (max 2 appearances per food)...")
print("=" * 60)
model_5day.optimize()

# Display results (summary)
if model_5day.status == GRB.OPTIMAL:
    print("OPTIMAL SOLUTION FOUND (5-Day Plan + Limited Repetition)")
    print("=" * 50)
    print(f"Minimum 5-Day Cost: ${model_5day.objVal:.2f}")
    for d in range(DAYS):
        print(f"\nDay {d+1} Food Quantities:")
        print("-" * 40)
        for i in range(NFOODS):
            if x_5day[d, i].x > 0.001:
                print(f"{foods[i]}: {x_5day[d, i].x:.3f} servings (${costs[i] * x_5day[d, i].x:.2f})")
    print("\nFood appearance counts over 5 days (should be ≤2):")
    for i in range(NFOODS):
        count = sum(appear[d, i].x > 0.5 for d in range(DAYS))
        if count > 0:
            print(f"{foods[i]}: {int(count)} days")
else:
    print(f"Optimization failed. Status: {model_5day.status}")
    if model_5day.status == GRB.INFEASIBLE:
        print("The problem is infeasible - no solution exists that satisfies all constraints.")
    elif model_5day.status == GRB.UNBOUNDED:
        print("The problem is unbounded - the objective can be improved indefinitely.")

In [ ]:
#GenAI Disclaimer: Code was co-created with GenAI (GitHub Copilot)
# Version 4: 5-Day Plan with Limited Repetition (max 2 appearances per food per 5 days)
# Relaxed food group constraints: min 1 serving/day for Fruit and Vegetable

import gurobipy as gp
from gurobipy import GRB

DAYS = 5
NFOODS = 49

# Decision variables: servings of food i on day d
model_5day = gp.Model("NutritionOptimization_5Day_LimitedRepetition")
x_5day = model_5day.addVars(DAYS, NFOODS, name="x", lb=0)

# Binary variables: 1 if food i is used on day d, 0 otherwise
appear = model_5day.addVars(DAYS, NFOODS, vtype=GRB.BINARY, name="appear")

# Objective: minimize total cost over 5 days
model_5day.setObjective(
    gp.quicksum(costs[i] * x_5day[d, i] for d in range(DAYS) for i in range(NFOODS)),
    GRB.MINIMIZE
)

# --- Per-day constraints ---
for d in range(DAYS):
    # Nutrient constraints
    for j in range(10):
        model_5day.addConstr(
            gp.quicksum(nutrition_data[i][j] * x_5day[d, i] for i in range(NFOODS)) >= nutrient_bounds[j][0],
            f"nutrient_{j}_min_day{d}"
        )
        model_5day.addConstr(
            gp.quicksum(nutrition_data[i][j] * x_5day[d, i] for i in range(NFOODS)) <= nutrient_bounds[j][1],
            f"nutrient_{j}_max_day{d}"
        )
    # Balanced diet constraints (AND logic)
    total_calories = gp.quicksum(nutrition_data[i][0] * x_5day[d, i] for i in range(NFOODS))
    total_protein = gp.quicksum(nutrition_data[i][5] * x_5day[d, i] for i in range(NFOODS))
    for i in range(NFOODS):
        model_5day.addConstr(
            nutrition_data[i][0] * x_5day[d, i] <= 0.3 * total_calories,
            f"balanced_calories_{i}_day{d}"
        )
        model_5day.addConstr(
            nutrition_data[i][5] * x_5day[d, i] <= 0.3 * total_protein,
            f"balanced_protein_{i}_day{d}"
        )
    # Food group constraints (relaxed: min 1 for Fruit/Vegetable)
    for group_name, indices in food_groups.items():
        if group_name in ["Fruit", "Vegetable"]:
            model_5day.addConstr(
                gp.quicksum(x_5day[d, i] for i in indices) >= 1,
                f"foodgroup_{group_name}_min1_day{d}"
            )
            model_5day.addConstr(
                gp.quicksum(x_5day[d, i] for i in indices) <= 4,
                f"foodgroup_{group_name}_max4_day{d}"
            )
        else:
            model_5day.addConstr(
                gp.quicksum(x_5day[d, i] for i in indices) >= 1,
                f"foodgroup_{group_name}_min1_day{d}"
            )

# --- Limited repetition constraint: each food appears at most 2 times in 5 days ---
for i in range(NFOODS):
    model_5day.addConstr(
        gp.quicksum(appear[d, i] for d in range(DAYS)) <= 2,
        f"limit_repetition_{i}"
    )

# --- Link x_5day and appear: if x_5day[d, i] > 0 then appear[d, i] = 1 ---
BIG_M = 1000  # Large enough upper bound for servings
for d in range(DAYS):
    for i in range(NFOODS):
        model_5day.addConstr(
            x_5day[d, i] <= BIG_M * appear[d, i],
            f"link_x_appear_{d}_{i}"
        )

print("SOLVING 5-DAY PLAN WITH LIMITED REPETITION (max 2 appearances per food)...")
print("=" * 60)
model_5day.optimize()

# Display results (summary)
if model_5day.status == GRB.OPTIMAL:
    print("OPTIMAL SOLUTION FOUND (5-Day Plan + Limited Repetition)")
    print("=" * 50)
    print(f"Minimum 5-Day Cost: ${model_5day.objVal:.2f}")
    for d in range(DAYS):
        print(f"\nDay {d+1} Food Quantities:")
        print("-" * 40)
        for i in range(NFOODS):
            if x_5day[d, i].x > 0.001:
                print(f"{foods[i]}: {x_5day[d, i].x:.3f} servings (${costs[i] * x_5day[d, i].x:.2f})")
    print("\nFood appearance counts over 5 days (should be ≤2):")
    for i in range(NFOODS):
        count = sum(appear[d, i].x > 0.5 for d in range(DAYS))
        if count > 0:
            print(f"{foods[i]}: {int(count)} days")
else:
    print(f"Optimization failed. Status: {model_5day.status}")
    if model_5day.status == GRB.INFEASIBLE:
        print("The problem is infeasible - no solution exists that satisfies all constraints.")
    elif model_5day.status == GRB.UNBOUNDED:
        print("The problem is unbounded - the objective can be improved indefinitely.")